In [26]:
import pandas as pd

data = pd.read_csv("test0.csv", sep=';')  # loading data
print(data.head()) 

                                             <Title>  \
0                 <Man killed in Kushtia road crash>   
1  <Truck-ambulance collision leaves one dead in ...   
2  <Lyricist Omar Faruk dies in Narsingdi road cr...   
3  <Three of a family killed in Kurigram road crash>   
4                <2 killed in Chattogram road crash>   

             <Publication Date> <Update Date>  \
0  <2018-10-16 13:37:00>,<None>  <Bangladesh>   
1         <2018-10-05 11:30:00>        <None>   
2         <2022-11-07 19:32:00>        <None>   
3         <2019-07-26 16:15:00>        <None>   
4         <2019-05-29 10:24:00>        <None>   

                                          <Location>  \
0  <https://www.unb.com.bd/category/Bangladesh/ma...   
1                                       <Bangladesh>   
2                                        <Narsingdi>   
3                                       <Bangladesh>   
4                                       <Bangladesh>   

                               

In [27]:
import spacy

nlp = spacy.load("en_core_web_sm")
import en_core_web_sm
nlp = en_core_web_sm.load()
doc = nlp("This is a sentence.")
print([(w.text, w.pos_) for w in doc])

[('This', 'PRON'), ('is', 'AUX'), ('a', 'DET'), ('sentence', 'NOUN'), ('.', 'PUNCT')]


In [30]:
import pandas as pd

data = pd.read_csv("test0.csv", sep=';', header=0)

# column names to verify
print(data.columns)


Index(['<Title>', '<Publication Date>', '<Update Date>', '<Location>',
       '<Link>', '<Raw Text>', '<HTML Text>'],
      dtype='object')


In [31]:
import pandas as pd
import spacy
from spacy.matcher import Matcher

# loading spaCy model
nlp = spacy.load("en_core_web_sm")

# matcher with desired patterns (setting it up)
matcher = Matcher(nlp.vocab)
vehicle_patterns = [
    [{"LOWER": "car"}],
    [{"LOWER": "truck"}],
    [{"LOWER": "motorcycle"}],
    # for now these patterns
]
matcher.add("VEHICLE", vehicle_patterns)

# function for extracting information
def extract_information(nlp, matcher, text):
    doc = nlp(text)
    places = [ent.text for ent in doc.ents if ent.label_ == "GPE"]
    times = [ent.text for ent in doc.ents if ent.label_ in ("DATE", "TIME")]
    vehicles = [doc[start:end].text for match_id, start, end in matcher(doc)]
    casualties = [ent.text for ent in doc.ents if ent.label_ == "CARDINAL" and "killed" in doc[ent.end:].text.lower()]
    injured = [ent.text for ent in doc.ents if ent.label_ == "CARDINAL" and "injured" in doc[ent.end:].text.lower()]
    
    return {
        "places": places,
        "times": times,
        "vehicles": vehicles,
        "casualties": casualties,
        "injured": injured
    }

# loading CSV file
csv_file_path = 'test0.csv'  # updating this path
data = pd.read_csv(csv_file_path, sep=';', on_bad_lines='skip')

# processing each row to extract information
extracted_data = []
for description in data['<Raw Text>']:  # updating '<Raw Text>' - contain descriptions
    info = extract_information(nlp, matcher, description.strip('<>'))
    extracted_data.append(info)

extracted_df = pd.DataFrame(extracted_data) # converting to dataframe

# saving extracted information to a CSV file
output_csv_path = 'test1.csv'  # creating new file 
extracted_df.to_csv(output_csv_path, index=False)
